In [2]:
import os
import json
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# 1. ĐỌC DỮ LIỆU GỐC
raw_data_path = "../Data/raw/bank-additional-full.csv"
df_raw = pd.read_csv(raw_data_path, sep=";")
print(f"[*] Kích thước dữ liệu gốc: {df_raw.shape[0]:,} dòng, {df_raw.shape[1]} cột")

# 2. LOẠI BỎ DÒNG TRÙNG LẶP (GIỮ NGUYÊN THỨ TỰ THỜI GIAN)
num_duplicates = df_raw.duplicated().sum()
print(f"[*] Số dòng trùng lặp hoàn toàn: {num_duplicates} dòng")

df = df_raw.drop_duplicates().reset_index(drop=True)
print(f"    -> Kích thước sau khi loại trùng: {df.shape[0]:,} dòng")

# 3. THỐNG KÊ 'unknown' (Sử dụng exclude='number' để hết warning pandas)
print("\n[*] Thống kê các cột có giá trị 'unknown':")
cat_cols = df.select_dtypes(exclude='number').columns.tolist()
unknown_stats = {}
for col in cat_cols:
    unknown_count = int((df[col] == 'unknown').sum())
    if unknown_count > 0:
        percent = (unknown_count / len(df)) * 100
        unknown_stats[col] = {"count": unknown_count, "percent": round(percent, 2)}
        print(f"    - Cột '{col:12}': {unknown_count:,} giá trị ({percent:.2f}%)")

# 4. CHIA TẬP 1: STRATIFIED SPLIT (Phân tầng theo nhãn y, random_state=42)
train_df, test_df = train_test_split(
    df, 
    test_size=0.2, 
    random_state=42, 
    stratify=df['y']
)

# Kiểm tra đảm bảo không có dòng nào giao thoa giữa train và test
overlap_check = pd.merge(train_df, test_df, how='inner')
assert len(overlap_check) == 0, "[!] Lỗi: Phát hiện dữ liệu trùng giữa Train và Test!"
print("\n[*] ĐÃ XÁC MINH (ASSERT): Train và Test hoàn toàn độc lập, không có dòng chung!")

# 5. CHIA TẬP 2: TIME-BASED SPLIT (Chia theo thứ tự thời gian gốc: 80% đầu làm Train, 20% sau làm Test)
split_idx = int(len(df) * 0.8)
time_train_df = df.iloc[:split_idx].copy()
time_test_df = df.iloc[split_idx:].copy()

print(f"[*] Phân chia theo thời gian (Time-based):")
print(f"    - Time-Train: {time_train_df.shape[0]:,} dòng | Tỷ lệ yes: {(time_train_df['y']=='yes').mean()*100:.2f}%")
print(f"    - Time-Test : {time_test_df.shape[0]:,} dòng | Tỷ lệ yes: {(time_test_df['y']=='yes').mean()*100:.2f}%")

# 6. LƯU TẤT CẢ FILE RA THƯ MỤC Data/processed/
processed_dir = "../Data/processed"
os.makedirs(processed_dir, exist_ok=True)

# Lưu Stratified Split
train_df.to_csv(os.path.join(processed_dir, "train.csv"), sep=";", index=False)
test_df.to_csv(os.path.join(processed_dir, "test.csv"), sep=";", index=False)

# Lưu Time-based Split
time_train_df.to_csv(os.path.join(processed_dir, "time_train.csv"), sep=";", index=False)
time_test_df.to_csv(os.path.join(processed_dir, "time_test.csv"), sep=";", index=False)

# Lưu Metadata split_info.json
split_info = {
    "total_original_samples": int(len(df_raw)),
    "duplicates_removed": int(num_duplicates),
    "clean_samples": int(len(df)),
    "features_count": int(df.shape[1] - 1),
    "target_column": "y",
    "stratified_split": {
        "train_samples": int(len(train_df)),
        "test_samples": int(len(test_df)),
        "train_yes_ratio": round((train_df['y'] == 'yes').mean() * 100, 2),
        "test_yes_ratio": round((test_df['y'] == 'yes').mean() * 100, 2),
        "random_state": 42
    },
    "time_based_split": {
        "train_samples": int(len(time_train_df)),
        "test_samples": int(len(time_test_df)),
        "train_yes_ratio": round((time_train_df['y'] == 'yes').mean() * 100, 2),
        "test_yes_ratio": round((time_test_df['y'] == 'yes').mean() * 100, 2)
    },
    "unknown_distribution": unknown_stats
}

with open(os.path.join(processed_dir, "split_info.json"), "w", encoding="utf-8") as f:
    json.dump(split_info, f, indent=4, ensure_ascii=False)

print("\n[*] ĐÃ XUẤT THÀNH CÔNG:")
print("    - train.csv & test.csv (Stratified)")
print("    - time_train.csv & time_test.csv (Time-based)")
print("    - split_info.json (Metadata chia dữ liệu)")

[*] Kích thước dữ liệu gốc: 41,188 dòng, 21 cột
[*] Số dòng trùng lặp hoàn toàn: 12 dòng
    -> Kích thước sau khi loại trùng: 41,176 dòng

[*] Thống kê các cột có giá trị 'unknown':
    - Cột 'job         ': 330 giá trị (0.80%)
    - Cột 'marital     ': 80 giá trị (0.19%)
    - Cột 'education   ': 1,730 giá trị (4.20%)
    - Cột 'default     ': 8,596 giá trị (20.88%)
    - Cột 'housing     ': 990 giá trị (2.40%)
    - Cột 'loan        ': 990 giá trị (2.40%)

[*] ĐÃ XÁC MINH (ASSERT): Train và Test hoàn toàn độc lập, không có dòng chung!
[*] Phân chia theo thời gian (Time-based):
    - Time-Train: 32,940 dòng | Tỷ lệ yes: 6.38%
    - Time-Test : 8,236 dòng | Tỷ lệ yes: 30.83%

[*] ĐÃ XUẤT THÀNH CÔNG:
    - train.csv & test.csv (Stratified)
    - time_train.csv & time_test.csv (Time-based)
    - split_info.json (Metadata chia dữ liệu)
